- **Purpose**: Purely for training and saving Keras models.
- **Content**: Load the Intel dataset, build the baseline `Conv2D` model, and train the 16 hyperparameter variations (layers, filters, pooling).
- **Output**: Save all the weights to a weights/ folder (e.g., `.h5` or `.weights.h5`).

In [1]:
%pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import itertools
import os
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models

In [6]:
BASE_DATA_PATH = "../../data/1_cnn_image_classification"
TRAIN_PATH = os.path.join(BASE_DATA_PATH, "seg_train", "seg_train")
TEST_PATH = os.path.join(BASE_DATA_PATH, "seg_test", "seg_test")

IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",  
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",  
)

class_names = train_dataset.class_names
NUM_CLASSES = len(class_names)
print(f"List kelas: {class_names}")
print(f"Jumlah kelas: {NUM_CLASSES}")

AUTOTUNE = tf.data.AUTOTUNE
train_dataset = (
    train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
)
val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)

Found 14034 files belonging to 6 classes.
Found 3000 files belonging to 6 classes.
List kelas: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
Jumlah kelas: 6


In [4]:
def build_cnn_model(
    num_layers, filter_configs, kernel_size, pool_type, input_shape=(128, 128, 3)
):
    model = models.Sequential()
    model.add(layers.InputLayer(input_shape=input_shape))
    model.add(layers.Rescaling(1.0 / 255))

    PoolingLayer = (
        layers.MaxPooling2D if pool_type == "max" else layers.AveragePooling2D
    )

    for i in range(num_layers):
        f_size = (
            filter_configs[i] if i < len(filter_configs) else filter_configs[-1]
        )

        model.add(
            layers.Conv2D(
                filters=f_size,
                kernel_size=kernel_size,
                padding="same",
                activation="relu",
            )
        )
        model.add(PoolingLayer(pool_size=(2, 2)))

    model.add(layers.Flatten())
    model.add(layers.Dense(64, activation="relu"))
    model.add(layers.Dense(NUM_CLASSES, activation="softmax"))

    macro_f1 = tf.keras.metrics.F1Score(average="macro", name="macro_f1")

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy", macro_f1],
    )

    return model

In [ ]:
num_layers_options = [2, 3]  
filters_options = [[32, 64, 128], [64, 128, 256]]  
kernel_size_options = [(3, 3), (5, 5)]  
pooling_options = ["max", "avg"]  

os.makedirs("saved_models", exist_ok=True)

combinations = list(
    itertools.product(
        num_layers_options, filters_options, kernel_size_options, pooling_options
    )
)

experiment_results = []
for idx, (n_layers, filters, k_size, p_type) in enumerate(
    combinations, start=1
):
    print("=" * 67)
    print(
        f"Eksperimen {idx}/16 | Layers: {n_layers} | Filters: {filters[:n_layers]} | Kernel: {k_size} | Pool: {p_type}"
    )
    print("=" * 67)

    model = build_cnn_model(n_layers, filters, k_size, p_type)
    
    history = model.fit(
        train_dataset, validation_data=val_dataset, epochs=4, verbose=1
    )

    val_f1 = history.history.get("val_macro_f1", [0])[-1]
    val_acc = history.history.get("val_accuracy", [0])[-1]

    weight_filename = f"saved_models/model_{idx}_L{n_layers}_F{filters[0]}_K{k_size[0]}_{p_type}.weights.h5"
    model.save_weights(weight_filename)

    experiment_results.append(
        {
            "id": idx,
            "layers": n_layers,
            "filters": str(filters[:n_layers]), 
            "kernel": k_size,
            "pooling": p_type,
            "val_macro_f1": val_f1,
            "val_accuracy": val_acc,
        }
    )